In [1]:
from pathlib import Path

import geopandas as gpd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [2]:
# Input and output paths
ROOT = Path().resolve().parent
RAW  = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed' 

In [3]:
raw_data = pd.read_csv(PROCESSED / "clean_dataset_with_permeability.csv")
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2525 entries, 0 to 2524
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   index                  2525 non-null   int64  
 1   well_id                2525 non-null   str    
 2   easting                2525 non-null   float64
 3   northing               2525 non-null   float64
 4   depth_tvd_m            2525 non-null   float64
 5   porosity_pct           2525 non-null   float64
 6   bulk_density_gcc       2525 non-null   float64
 7   formation_thickness_m  2525 non-null   float64
 8   distance_to_usp_km     2525 non-null   float64
 9   temp_harrison          2525 non-null   float64
 10  temp_res_c             2525 non-null   float64
 11  permeability_md        2525 non-null   float64
 12  k_log                  2525 non-null   float64
dtypes: float64(11), int64(1), str(1)
memory usage: 256.6 KB


In [4]:
df = raw_data.copy()
well_list = df["well_id"].unique().tolist()

In [5]:
wells = {
    row["well_id"]: {
        "Thickness": row["formation_thickness_m"],
        "Permeability": row["permeability_md"] * 9.869e-16,
        "Transmissivity": row["formation_thickness_m"] * row["permeability_md"] * 9.869e-16,
        "Temperature": row["temp_res_c"],
    }
    for _, row in df.iterrows()
}

### Flow Rate

$$Q=\frac{2\pi T\Delta P}{\mu \ln (r_{e}/r_{w})}$$

In [6]:
for well in well_list:
    T = wells[well]["Transmissivity"]
    delta_p = 18 * 1e5
    mu = 2.414e-5 * 10 ** (247.8 / (wells[well]["Temperature"] + 133.15))
    r_e = 1000
    r_w = 8 * 0.0254 / 2

    Q = (2 * np.pi * T * delta_p) / (mu * np.log(r_e / r_w))
    wells[well]["Flow Rate"] = Q

### Thermal Power

$$ P_{th} = \dot{m}C_p(T_p - T_r)$$

In [7]:
for well in well_list:
    rho_water = 970   # kg/m3
    m_dot = rho_water * wells[well]["Flow Rate"]  # kg/s

    cp_water = 4186   # J/(kg*K)
    T_p = wells[well]["Temperature"]        # deg C
    T_r = 40                    # deg C

    P_th = m_dot * cp_water * (T_p - T_r)   # Watts
    P_th_MW = P_th / 1e6

    wells[well]["Thermal Power"] = P_th_MW

In [8]:
wells

{'BLT-01': {'Thickness': 122.37544947053402,
  'Permeability': 8.092579999999999e-14,
  'Transmissivity': 9.903331148762541e-12,
  'Temperature': 79.66670862465342,
  'Flow Rate': np.float64(0.0345611026743731),
  'Thermal Power': np.float64(5.56653205807582)},
 'JUT-01': {'Thickness': 125.75189999999998,
  'Permeability': 3.9476e-14,
  'Transmissivity': 4.9641820043999985e-12,
  'Temperature': 71.27916626642774,
  'Flow Rate': np.float64(0.015519597084710438),
  'Thermal Power': np.float64(1.9710905186827294)}}